# Spectrograph Demo

In addition to supporting simulations with that apply filters to provide bandfluxes, LightCurveLynx also provides the ability to generate spectra at different time steps. In contrast to band flux densities produced by passbands, the spectrograph output is the measured (bin-integrated) flux with units of erg/s/cm².

## Spectrograph

The core class used for generating spectra is the `Spectrograph` class, which defines information on the bins and sensitivity of the instrument. A user defines a spectrograph instrument by passing an array of bin centers (wavelength in Angstroms) and an optional array of per-bin scales.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lightcurvelynx.astro_utils.spectrograph import Spectrograph

wave_min = np.arange(4000, 8000, 100)
wave_max = np.arange(4100, 8100, 100)
my_spectrograph = Spectrograph(wave_min, wave_max)

The spectrograph's `evaluate()` function is then used to turn a model's SED into spectrograph readings with units erg/s/cm².

Let's look at a concrete function where we evaluate the spectrograph of [sncosmo](https://sncosmo.readthedocs.io/en/stable/)'s SALT2 model.

In [ ]:
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel

model = SncosmoWrapperModel(
    "salt2-h17",  # Model name
    t0=53370.5,
    x0=10 ** (-0.4 * (22.0 - 10.635)),  # x0 is set to match a peak magnitude of 22.0 in the B-band.
    x1=0.01,
    c=0.001,
    ra=0.0,
    dec=0.0,
    redshift=0.3,
    node_label="source",
)

We can evaluate the modeled SEDs at a few times (2 days before t0, t0, 2 days after t0, 5 days after t0). We use the bin's wavelength centers as our evaluation points.

In [ ]:
times = 53370.5 + np.array([-15.0, 0.0, 15.0, 30.0])
seds = model.evaluate_sed(times, my_spectrograph.bin_centers)
spectra = my_spectrograph.evaluate(seds)

plt.plot(my_spectrograph.bin_centers, spectra[0], marker=".", linestyle=":", label="t=-5.0")
plt.plot(my_spectrograph.bin_centers, spectra[1], marker=".", linestyle=":", label="t=0.0")
plt.plot(my_spectrograph.bin_centers, spectra[2], marker=".", linestyle=":", label="t=5.0")
plt.plot(my_spectrograph.bin_centers, spectra[3], marker=".", linestyle=":", label="t=10.0")
plt.xlabel("Wavelength (Angstrom)")
plt.ylabel("Measured Flux (erg/s/cm²)")
plt.title("Spectral Energy Distribution of SN at Different Times")
_ = plt.legend()

## Using in Simulations 

We can include spectrograph readings into our normal simulation workflow by adding an `ObsTable` with the spectrograph's pointings and a "PassbandGroup" that is just the spectrograph instrument itself. While any `ObsTable` subclass can be used (all that matters is the pointing information), we have provided a minimal `SpectrographObsTable` that uses a tighter radius and does not require zero point or filter information.

We start by creating a fake observation table that includes points at our object and away from our object.

In [ ]:
from lightcurvelynx.obstable.spectrograph_table import SpectrographObsTable

data = {
    "ra": [0.0, 10.0, 0.0, 0.0, 10.0, 0.0],
    "dec": [0.0, -10.0, 0.0, 0.0, -10.0, 0.0],
    "time": 53370.5 + np.array([-15.0, -12.0, 0.0, 15.0, 20.0, 30.0]),
    "exptime": [1800.0, 1800.0, 1800.0, 1800.0, 1800.0, 1800.0],
}
obstable = SpectrographObsTable(data)

As of version 0.4.0, we wrap all the survey related information in a ``SurveyInfo`` object.

In [ ]:
from lightcurvelynx.survey_info import SurveyInfo

survey_info = SurveyInfo(
    obstable=obstable,
    passbands=my_spectrograph,  # The spectrograph is used as the passband information for the survey.
    survey_name="spectrograph",
)

We can then just pass the model, survey information, and spectrograph to the simulation function.

In [ ]:
from lightcurvelynx.simulate import simulate_lightcurves

results = simulate_lightcurves(
    model,  # The model we are simulating.
    1,  # The number of simulations to run,
    survey_info,  # The survey information, which includes the observation times and passbands.
)
results.head()

The results look like the table for a normal simulation, but contain an additional "spectra" nested column. Each row of the nested data contains the time, instrument, list of wavelengths, and list of measured values.

Out of the six observations in our fake table, four of them part pointed at the object. So the results include 4 spectra.

In [ ]:
row_0_spectra_table = results.iloc[0]["spectra"]
row_0_spectra_table

We can plot the resulting spectra. Note that they are in different units (F_lambda in erg/s/cm²) than the original SED from the model.

In [ ]:
row_0_spectra_table = results.iloc[0]["spectra"]
for row_idx in range(len(row_0_spectra_table)):
    plt.plot(
        row_0_spectra_table.iloc[row_idx]["waves"],
        row_0_spectra_table.iloc[row_idx]["measured_flux"],
        marker=".",
        linestyle=":",
        label=f"t={row_0_spectra_table.iloc[row_idx]['mjd']:.2f}",
    )
plt.xlabel("Wavelength (Angstrom)")
plt.ylabel("Measured Flux (erg/s/cm²)")
plt.title("Spectral Energy Distribution of SN at Different Times")
_ = plt.legend()

## Noise Model

We can simulate spectral noise using the `SNANANoiseModel`, which produces Poisson noise as implemented in SNANA. We first initialize a spectrograph by reading in an SNANA spectrograph file using `Spectrograph.from_snana_file()`, and define the noise model using the same file. Then we wrap them using `SurveyInfo()`. We can now generate noisy spectra using `simulate_lightcurves()`. We use `4MOST_SPECTROGRAPH.DAT` as an example, which is available as part of SNDATA_ROOT for SNANA (Richard Kessler, Dillon Brout and David Jones (2026) “SNDATA_ROOT for SNANA software”. Zenodo. Available at: [https://doi.org/10.5281/zenodo.23934400](https://doi.org/10.5281/zenodo.23934400)).

In [ ]:
from lightcurvelynx import _LIGHTCURVELYNX_BASE_DATA_DIR
from lightcurvelynx.noise_models.spectrograph_noise_models import SNANANoiseModel

snana_spectrograph_file = _LIGHTCURVELYNX_BASE_DATA_DIR / "spectrograph_example" / "4MOST_SPECTROGRAPH.DAT"
my_spectrograph2 = Spectrograph.from_snana_file(snana_spectrograph_file)
noise_model = SNANANoiseModel(snana_spectrograph_file)
survey_info2 = SurveyInfo(
    obstable=obstable,
    passbands=my_spectrograph2,  # The spectrograph is used as the passband information for the survey.
    noise_model=noise_model,  # The noise model to be used for the survey.
    survey_name="spectrograph",
)

results = simulate_lightcurves(
    model,  # The model we are simulating.
    1,  # The number of simulations to run,
    survey_info2,  # The survey information, which includes the observation times and passbands.
)
results.head()

Now let's plot the spectra

In [ ]:
row_0_spectra_table = results.iloc[0]["spectra"]
for row_idx in range(len(row_0_spectra_table)):
    plt.plot(
        row_0_spectra_table.iloc[row_idx]["waves"],
        row_0_spectra_table.iloc[row_idx]["measured_flux"],
        label=f"t={row_0_spectra_table.iloc[row_idx]['mjd']:.2f}",
    )
plt.xlabel("Wavelength (Angstrom)")
plt.ylabel("Measured Flux (erg/s/cm²)")
plt.title("Spectral Energy Distribution of SN at Different Times")
_ = plt.legend()

Now let's redefine a `SpectrographObsTable` so the exposure times are three times the exposure time used above. We can see the effect of different exposure times.

In [ ]:
data["exptime"] = np.array(data["exptime"]) * 3.0
obstable2 = SpectrographObsTable(data)
survey_info2 = SurveyInfo(
    obstable=obstable2,
    passbands=my_spectrograph2,  # The spectrograph is used as the passband information for the survey.
    noise_model=noise_model,  # The noise model to be used for the survey.
    survey_name="spectrograph",
)

results = simulate_lightcurves(
    model,  # The model we are simulating.
    1,  # The number of simulations to run,
    survey_info2,  # The survey information, which includes the observation times and passbands.
)
row_0_spectra_table = results.iloc[0]["spectra"]
for row_idx in range(len(row_0_spectra_table)):
    plt.plot(
        row_0_spectra_table.iloc[row_idx]["waves"],
        row_0_spectra_table.iloc[row_idx]["measured_flux"],
        label=f"t={row_0_spectra_table.iloc[row_idx]['mjd']:.2f}",
    )
plt.xlabel("Wavelength (Angstrom)")
plt.ylabel("Measured Flux (erg/s/cm²)")
plt.title("Spectral Energy Distribution of SN at Different Times")
_ = plt.legend()